## Option 1: XAI via SDGRS XAI CLI

* **Input** (`--input-path`): csv abstracts in the format of ZO_up
* **Output** (`--output-path`): csv abstracts with token_scores column (n_tokens x n_labels) that contain the feature attribution values of a selected XAI method

In [1]:
!sdgrs-xai-cli \
    --model-family=scibert \
    --model-path=$HOME/.cache/huggingface/checkpoints/sdg-scibert/allenai/scibert_scivocab_cased-zo_up/checkpoint-553 \
    --method=shap-partition \
    --input-path ../experiments/data/zo_up.csv \
    --output-path ./zo_up_scibert_xai.csv \
    explain

2025-04-28 04:41:20 [info     ] Parsed arguments, loading model and tokenizer...
2025-04-28 04:41:21 [info     ] Loaded model and tokenizer     model_family=scibert
2025-04-28 04:41:21 [info     ] Data manager set up            data_source_and_target=local
2025-04-28 04:41:21 [info     ] Starting explain action        method=shap-partition
2025-04-28 04:41:24 [info     ] Starting to process documents 
2025-04-28 04:41:24 [info     ] Setting up worker              device=device(type='cuda', index=0) real_gpu_id=0 worker_id=0
2025-04-28 04:43:50 [info     ] Progress update                total_processed=100 total_queued=901
2025-04-28 04:46:10 [info     ] Progress update                total_processed=200 total_queued=901
2025-04-28 04:48:10 [info     ] Progress update                total_processed=300 total_queued=901
2025-04-28 04:50:12 [info     ] Progress update                total_processed=400 total_queued=901
2025-04-28 04:52:38 [info     ] Progress update                total_p

# Option 2: XAI via the SDGRS XAI lib

In [2]:
import torch
from sdgrs_xai.models import setup_model_and_tokenizer
from sdgrs_xai.explainers import get_explainer
from pathlib import Path

XAI_METHOD = "shap-partition" # one of app.explainers.model.ExplainerMethod
MODEL_FAMILY = "scibert"
MODEL_PATH = Path.home() / ".cache/huggingface/checkpoints/sdg-scibert/allenai/scibert_scivocab_cased-zo_up/checkpoint-553"

model, tokenizer = setup_model_and_tokenizer(
    model_family=MODEL_FAMILY,
    method=XAI_METHOD,
    model_path=MODEL_PATH
)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)

explainer = get_explainer(
    method_name=XAI_METHOD,
    model=model,
    tokenizer=tokenizer,
    device=device,
    model_family=MODEL_FAMILY,
)

In [3]:
from sdgrs_xai.explainers.model import XAIOutput

xai_output: XAIOutput = explainer.explain(abstract="Is this about clean energy?")
print(xai_output)

XAIOutput(text='Is this about clean energy?', input_tokens=['', 'Is ', 'this ', 'about ', 'clean ', 'energy', '?', ''], token_scores=[[-3.3760443329811096e-09, 1.0273652151226997e-08, -1.3620592653751373e-08, 1.4551915228366852e-10, -2.153683453798294e-09, 0.0, 1.7881393432617188e-07, 4.976755008101463e-09, 2.3283064365386963e-09, 5.8353180065751076e-09, 7.508788257837296e-09, -1.6205012798309326e-07, -7.799826562404633e-09, -7.712515071034431e-09, 6.984919309616089e-10, -2.240994945168495e-09, -1.979060471057892e-09], [-0.0028195245831739157, -0.020310675346991047, -0.0009451706428080797, -0.0067111839016433805, 0.009905767801683396, -0.0064418804831802845, 0.18990272376686335, 0.0018230639689136297, 0.02421884937211871, -0.03765299315273296, 0.014594980981200933, -0.13871571142226458, 0.0026518674567341805, -0.004716464056400582, -0.018241424986626953, -0.006752095447154716, 0.00020989496260881424], [0.001011320942780003, -0.0028466495277825743, -0.0074882941553369164, 0.007889327156

In [4]:
# xai_output.token_scores are of shape [n_tokens, n_labels]
# therefore this score corresponds to the token "energy"
xai_output.token_scores[5][xai_output.predicted_id]


0.20644421671750024